# Train Arm B (blt_entropy_patching) at 20000 steps / 0.5GB (scale-up x4, question 1.3)

See `plans/PLAN.md` question 1.3, `phases/phase-2-small-train.md`. Checking whether the
A-vs-{B,C} gap (widening at 5000 steps) keeps widening, plateaus, or reverses with 4x
more training. Same model size/entropy_pretrain_steps as the 5000-step run - only
max_steps and dataset size change (single variable).


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os

_candidates = [
    "/kaggle/input/vislm-research-code",
    "/kaggle/input/datasets/nguyennn263/vislm-research-code",
]
CODE_DIR = next(p for p in _candidates if os.path.isdir(p))
os.environ["PYTHONPATH"] = CODE_DIR
print("CODE_DIR:", CODE_DIR)


In [ ]:
!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python {CODE_DIR}/setup/download_prepare_data.py \
  --target-gb 0.5 --out-dir /kaggle/working/data/prepared/fineweb2_vi --shard-size-mb 100


In [ ]:
!python -m vislm.train {CODE_DIR}/pillar1_configs/1_3_arm_B_blt.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_B_20000 \
  train.max_steps=20000 train.warmup_steps=400 \
  train.save_every=10000 train.eval_every=1000 train.entropy_pretrain_steps=500


In [ ]:
import json

train_losses, val_losses = [], []
with open("/kaggle/working/runs/arm_B_20000/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            train_losses.append((row["step"], row["loss"]))
        if "eval_step" in row:
            val_losses.append((row["eval_step"], row["val_loss"]))

summary = {
    "n_steps": len(train_losses),
    "first_loss": train_losses[0][1],
    "last_loss": train_losses[-1][1],
    "min_loss": min(l for _, l in train_losses),
    "last_20_avg": sum(l for _, l in train_losses[-20:]) / len(train_losses[-20:]),
    "val_loss_curve": val_losses,
}
print(summary)

with open("/kaggle/working/metrics_arm_B_20000.jsonl", "w") as f:
    f.write(json.dumps({"section": "arm_B_20000", "results": summary}) + "\n")
